In [34]:
import numpy as np
from fluid_property import fluids_dict

def score_fluid_mass(mass_kg):
    """
    Calculates a score (0-5) for fluid mass (kg), where 0.05 kg is 5 and 0.1 kg is 0.
    The 0.05 kg to 0.1 kg range is divided into 6 discrete levels (0.01 kg per level).
    """

    # --- Inputs outside the defined scoring range are handled logically ---
    if mass_kg < 0.05:
        # Mass is better than the best-case threshold
        return 5
    if mass_kg >= 0.10:
        # Mass is worse than or equal to the worst-case threshold
        return 0

    # --- Core Scoring Logic (0.01 kg segments) ---
    if mass_kg < 0.06:
        return 5
    elif mass_kg < 0.07:
        return 4
    elif mass_kg < 0.08:
        return 3
    elif mass_kg < 0.09:
        return 2
    elif mass_kg < 0.10:
        return 1
    else:
        # Should be covered by the initial check, but included for completeness
        return 0

def score_pump_power(power_W):
    # --- Core Scoring Logic ---
    if power_W < 1e-6:
        return 5
    elif power_W < 1e-5:
        return 4
    elif power_W < 1e-4:
        return 3
    elif power_W < 1e-3:
        return 2
    elif power_W < 1e-2:
        return 1
    else:
        # Should be covered by the initial check, but included for completeness
        return 0

def calc_power(fluid, Q, delT, L, D):
    """
    calculate fluid power delivered. Pump efficiency not considered.
    """
    # fluid parameters
    cp    = fluid['cp']     # heat capacity  J/(kg K)
    rho_f = fluid['rho_f']  # fluid density  kg/m3
    mu    = fluid['mu']     # dynamic viscosity (Pa s)
    
    # flow calculations
    mt = Q / (cp*delT)       # mass flow rate
    Vt = mt / rho_f          # volumetric flow rate
    Ac = 0.25*np.pi*D*D      # pipe inner cross section area
    v = mt / (rho_f*Ac)      # flow speed
    Re = rho_f * v * D / mu  # Reynolds Number

    # find friction factor
    if Re < 2300:
        print(f"laminar, Re = {Re}")
        f = 64/Re
    elif Re >= 4000:
        print(f"Turbulent, Re = {Re}")
        f = (0.79*np.log(Re)-1.64)**(-2)
    else:
        print("undefined Re.")
    
    # =========================
    # ===== Pressure Loss =====
    # =========================
    # pressure loss from friction in the straight pipe 
    delP_pipe = f * (L / D) * (0.5*rho_f*v*v)
    # pressure loss from components
    N_elbow = 10   # number of 90° elbows
    L_D_elbow = 30 # equivalent L/D of 90° elbows  /*https://tameson.com/pages/pressure-drop*/
    delP_comp = f * (N_elbow * L_D_elbow) * (0.5*rho_f*v*v)  # pressure loss from components (gamma * 0.5*rho_f*v*v, gamma is resistance coefficient by test or vendor specification)
    # total pressure loss
    delP_tot = delP_pipe + delP_comp
    
    # power needed to be delivered to fluid
    P_f = delP_tot * Vt
    
    # fluid mass
    mass_f = Ac * L * rho_f
    
    # print
    print(f"volume flow rate: {Vt*3600e6:.4f} mL/h")
    print(f"mass flow rate: {mt*3600:.4f} kg/h")
    print(f"total pressure drop: {delP_tot:.4f} Pa  ({delP_comp/delP_tot*100:.1f} % from components)")
    print(f"pump power needed: {P_f:.3e} W       score: {score_pump_power(P_f)}")
    print(f"total fluid mass: {mass_f:.4f} kg       score: {score_fluid_mass(mass_f)}")
    
    return P_f

def mass_frac_from_volume_frac(volume_frac):
    """
    calculate mass fraction of Ethylene Glycol from its volume fraction
    """
    rho_eg = 1113.
    rho_h2o = 1000.
    return rho_eg/rho_h2o*volume_frac

def Q_radiate(emissivity, A, T):
    sigma = 5.67e-8
    T_space = 223
    Q = sigma * emissivity * A * (T**4 - T_space**4)
    return Q

def PTR_heat_ratio(Ta, Tc, p_ratio, kappa):
    return Ta / Tc * p_ratio**((kappa-1)/kappa)

def calc_fluids():
    # system parameters
    Q = 40   # Heat load from PTR (W)
    delT = 50  # temperature change of fluid (K)
    L = 2     # total pipe length (m)
    D = 0.00635   # Inner Pipe Diameter (m)
    
    for key, value in fluids_dict.items():
        print("\n==================================")
        print(key)
        calc_power(fluids_dict[key], Q, delT, L, D)

if __name__ == "__main__":
    calc_fluids()


imag_fluid
laminar, Re = 44.74424870855732
volume flow rate: 1920.0000 mL/h
mass flow rate: 1.9200 kg/h
total pressure drop: 124.7334 Pa  (48.8 % from components)
pump power needed: 6.652e-05 W       score: 3
total fluid mass: 0.0633 kg       score: 4

CFC11
laminar, Re = 243.74995264314958
volume flow rate: 2139.3477 mL/h
mass flow rate: 3.4270 kg/h
total pressure drop: 45.5377 Pa  (48.8 % from components)
pump power needed: 2.706e-05 W       score: 3
total fluid mass: 0.1015 kg       score: 0

Therminol59
laminar, Re = 0.0438717983402581
volume flow rate: 1924.4905 mL/h
mass flow rate: 1.9726 kg/h
total pressure drop: 131004.9143 Pa  (48.8 % from components)
pump power needed: 7.003e-02 W       score: 0
total fluid mass: 0.0649 kg       score: 4

TherminolLT
laminar, Re = 26.27616944652121
volume flow rate: 2043.8143 mL/h
mass flow rate: 1.8824 kg/h
total pressure drop: 221.6655 Pa  (48.8 % from components)
pump power needed: 1.258e-04 W       score: 2
total fluid mass: 0.0583 kg   

In [57]:
Q_radiate(emissivity = 0.5, A=1456/10000, T = 298)

22.344295390326

In [7]:
PTR_heat_ratio(298, 78, 2, 1.4)

4.657257294268381

In [4]:
Q=40
sigma = 5.67e-8
emis = 0.5
T = 249.5
A = Q / (sigma*emis*T**4)
A

0.36410339485547816